In [0]:
%sql
use catalog deltalake_catalog;

In [0]:
landing_zone = "/Volumes/deltalake_catalog/default/raw"
orders_data = landing_zone + "/ordershistory"
checkpoint_path = landing_zone + "/orders_checkpoint"

In [0]:
from pyspark.sql.functions import col, current_timestamp
ordersdf = (spark.readStream 
.format("cloudFiles") 
.option("cloudFiles.format", "csv") 
.option("cloudFiles.inferSchema", "true") 
.option("cloudFiles.inferColumnTypes", "true") 
.option("cloudFiles.schemaHints","order_id BIGINT, order_date string")
.option("cloudFiles.schemaLocation", checkpoint_path) 
.load(orders_data)
.withColumn("file_name", col("_metadata.file_path"))
.withColumn("ingesttime", current_timestamp())

)




In [0]:
%sql
drop table if exists deltalake_catalog.default.ordersdelta;

In [0]:
ordersdf.writeStream \
.format("delta") \
.option("checkpointLocation", checkpoint_path) \
.option("mergeSchema", True) \
.outputMode("append") \
.trigger(availableNow=True) \
.toTable("deltalake_catalog.default.ordersdelta")



In [0]:
%sql
select * from deltalake_catalog.default.ordersdelta;

order_id,order_date,customer_id,order_status,_rescued_data,file_name,ingesttime,order_amount
5555555,2013-07-25 00:00:00.0,11599,CLOSED,null,/Volumes/deltalake_catalog/default/raw/ordershistory/orders4.csv,2025-10-17T08:43:50.056Z,null
6666666,2013-07-25 00:00:00.0,256,PENDING_PAYMENT,null,/Volumes/deltalake_catalog/default/raw/ordershistory/orders4.csv,2025-10-17T08:43:50.056Z,null
7777777,2013-07-25 00:00:00.0,null,COMPLETE,"{""customer_id"":""COMPLETE"",""_file_path"":""/Volumes/deltalake_catalog/default/raw/ordershistory/orders4.csv""}",/Volumes/deltalake_catalog/default/raw/ordershistory/orders4.csv,2025-10-17T08:43:50.056Z,null
8888888,2013-07-25 00:00:00.0,8827,CLOSED,null,/Volumes/deltalake_catalog/default/raw/ordershistory/orders4.csv,2025-10-17T08:43:50.056Z,null
100000,2013-07-25 00:00:00.0,11599,CLOSED,null,/Volumes/deltalake_catalog/default/raw/ordershistory/orders3.csv,2025-10-17T08:42:52.985Z,10
200000,2013-07-25 00:00:00.0,256,PENDING_PAYMENT,null,/Volumes/deltalake_catalog/default/raw/ordershistory/orders3.csv,2025-10-17T08:42:52.985Z,20
300000,2013-07-25 00:00:00.0,12111,COMPLETE,null,/Volumes/deltalake_catalog/default/raw/ordershistory/orders3.csv,2025-10-17T08:42:52.985Z,30
400000,2013-07-25 00:00:00.0,8827,CLOSED,null,/Volumes/deltalake_catalog/default/raw/ordershistory/orders3.csv,2025-10-17T08:42:52.985Z,40
1111111,2013-07-25 00:00:00.0,11599,CLOSED,null,/Volumes/deltalake_catalog/default/raw/ordershistory/orders1.csv,2025-10-17T08:41:29.093Z,null
1111111,2013-07-25 00:00:00.0,256,PENDING_PAYMENT,null,/Volumes/deltalake_catalog/default/raw/ordershistory/orders1.csv,2025-10-17T08:41:29.093Z,null
